# Cloud training: LoL Ability VFX Detector

Train a champion's ability classifier on a cloud GPU (Colab / runpod / etc.).

**What you need to provide:**
- This repo (clone from your Git remote, or upload it).
- Your `data/clips/<champion>/` folder, *or* the raw videos + annotations so the
  builder can regenerate clips here.

Steps below: install deps -> get data -> build clips (optional) -> train ->
download the checkpoint back to your machine into `models/<champion>/best.pt`.

In [ ]:
# 1. Get the code. Option A: clone your remote.
REPO_URL = ""  # e.g. https://github.com/you/league-ability-detector.git
BRANCH = "main"
CHAMPION = "ezreal"

import os
if REPO_URL:
    !git clone --branch $BRANCH $REPO_URL repo
    %cd repo
else:
    print("No REPO_URL set. Upload the repo manually and %cd into it.")

In [ ]:
# 2. Install deps. On a CUDA box, install the matching torch build first.
!pip install -q torch torchvision  # use the CUDA index-url on GPU boxes
!pip install -q -r requirements.txt
# x3d_s backbone is fetched via torch.hub and needs pytorchvideo:
!pip install -q pytorchvideo

In [ ]:
# 3. Get your data. Easiest: upload a zip of data/ and unzip it here.
#    Expected layout: data/raw_videos, data/annotations, (optional) data/clips
# from google.colab import files; up = files.upload()  # then unzip below
# !unzip -o data.zip -d .
!ls -R data | head -40

In [ ]:
# 4. (Optional) Build clips from raw videos + annotations.
#    Skip if you uploaded a prebuilt data/clips/<champion>/ folder.
!python -m src.dataset.build --config configs/$CHAMPION.yaml

In [ ]:
# 5. Train. Device auto-detects CUDA.
!python -m src.train.train --config configs/$CHAMPION.yaml --device auto

In [ ]:
# 6. Download the checkpoint back to your machine.
ckpt = f"models/{CHAMPION}/best.pt"
print("checkpoint:", ckpt, os.path.getsize(ckpt), "bytes")
# from google.colab import files; files.download(ckpt)